In [1]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

c:\Users\bpu320145\2026_Projects\Agentic_Chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. State Definition
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [3]:
# 2. Tool and LLM Setup
def sensitive_tool(query: str) -> str:
    """A tool that requires user approval before executing, such as writing to a database."""
    return f"Successfully executed sensitive operation for: {query}"

tools = [sensitive_tool]
llm = ChatOpenAI(model="gpt-4o-mini").bind_tools(tools)


In [4]:
# 3. Node Definition
def agent_node(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

In [8]:
# 4. Graph Construction
builder = StateGraph(State)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(tools=tools))

builder.add_edge(START, "agent")
# tools_condition routes to "tools" if a tool is called, otherwise END
builder.add_conditional_edges("agent", tools_condition, {"tools": "tools", END: END})
builder.add_edge("tools", "agent")

# 5. Compilation with Interruption
memory = MemorySaver()
# The graph will halt execution right before entering the "tools" node
graph = builder.compile(checkpointer=memory, interrupt_before=["tools"])

In [9]:
# 6. Execution - Step 1: Initial Run
config = {"configurable": {"thread_id": "hitl_test_1"}}
user_input = "Please run the sensitive tool for the query 'delete old logs'."

print("--- Step 1: Initiating Graph ---")
for chunk in graph.stream({"messages": [HumanMessage(content=user_input)]}, config=config, stream_mode="updates"):
    print(chunk)

--- Step 1: Initiating Graph ---
{'agent': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 66, 'total_tokens': 83, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_6d6670c036', 'id': 'chatcmpl-DDuo0tgfMzGanpLNZrfIyposLUn9g', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c9fe3-4eb7-79b2-a0b6-341ddb365dd5-0', tool_calls=[{'name': 'sensitive_tool', 'args': {'query': 'delete old logs'}, 'id': 'call_q9wq0fSMnajywg4BtKK2CR3v', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 66, 'output_tokens': 17, 'total_tokens': 83, 'input_token_details': {'audio': 0, 'cache_read': 0},

In [10]:
# 7. Execution - Step 2: Inspecting the Paused State
state = graph.get_state(config)
print("\n--- Step 2: Graph Paused ---")
print(f"Pending Next Node: {state.next}")

if state.next == ('action',):
    pending_tool_call = state.values["messages"][-1].tool_calls[0]
    print(f"The LLM wants to call: {pending_tool_call['name']}")
    print(f"With arguments: {pending_tool_call['args']}")
    
    # 8. Execution - Step 3: Resuming
    # In a real application, you would wait for UI button click here.
    user_approval = input("\nApprove this tool execution? (y/n): ")
    
    if user_approval.lower() == 'y':
        print("\n--- Step 3: Resuming Graph ---")
        # Passing None as input tells LangGraph to resume from the last checkpoint
        for chunk in graph.stream(None, config=config, stream_mode="updates"):
            print(chunk)
    else:
        print("\nExecution cancelled by user.")


--- Step 2: Graph Paused ---
Pending Next Node: ('tools',)


In [11]:
# --- Step 4: Intercept and Modify State ---
print("\n--- Step 4: Modifying State Before Resuming ---")

# Fetch the current paused state
state = graph.get_state(config)
last_message = state.values["messages"][-1]

if last_message.tool_calls:
    # 1. Extract and modify the tool call arguments
    modified_tool_calls = last_message.tool_calls.copy()
    modified_tool_calls[0]["args"]["query"] = "archive old logs"
    
    # 2. Apply the modification to the message object
    last_message.tool_calls = modified_tool_calls

    # 3. Push the updated state back into the checkpointer
    # as_node="agent" tells the graph to treat this manual update as if the agent produced it
    graph.update_state(config, {"messages": [last_message]}, as_node="agent")
    
    print(f"Tool arguments manually altered to: {modified_tool_calls[0]['args']}")


--- Step 4: Modifying State Before Resuming ---
Tool arguments manually altered to: {'query': 'archive old logs'}


In [12]:
# --- Step 5: Resume Graph with New Data ---
print("\n--- Step 5: Resuming Execution ---")
# Passing None tells LangGraph to pick up from the pause point, using the new altered state
for chunk in graph.stream(None, config=config, stream_mode="updates"):
    print(chunk)


--- Step 5: Resuming Execution ---
{'tools': {'messages': [ToolMessage(content='Successfully executed sensitive operation for: archive old logs', name='sensitive_tool', id='371127fc-79e1-4dc0-bceb-81e7bf8a968e', tool_call_id='call_q9wq0fSMnajywg4BtKK2CR3v')]}}
{'agent': {'messages': [AIMessage(content='I have successfully executed the sensitive operation to archive old logs. If you need further assistance, feel free to ask!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 101, 'total_tokens': 125, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_6d6670c036', 'id': 'chatcmpl-DDv2ELmNvFvoeaadw0LYAz0Iv38ff', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None